In [1]:
import os

# Adjust the path to match your Box sync folder location
box_path = os.path.expanduser(r"C:\Users\HIALAB\Box\Human_AGV_project\ISU_Modeling\Code")

In [113]:
# base libraries
import numpy as np
import pandas as pd

# plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# regular expression module in python to find all sequences of digits in a given string
import re

# data and preprocessing libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import preprocessing

import os
from datetime import datetime, timedelta
import math
from scipy import stats
from scipy.stats import ttest_ind
from scipy.interpolate import griddata
from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import RBFInterpolator
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

# Error evaluation libraries
from sklearn.metrics import confusion_matrix

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Error evaluation libraries
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_absolute_percentage_error as mape
from sklearn.metrics import r2_score
from sklearn.metrics import classification_report
from sklearn.metrics import make_scorer

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.utils.validation")

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder

from tabulate import tabulate

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate, TimeDistributed
from tensorflow.keras.models import Model

In [5]:
import sys
print(sys.executable)

C:\Users\HIALAB\anaconda3\envs\tf_env\python.exe


In [11]:
# Specify the directory and filename
save_dir = "./Images/"

In [13]:
study_data = pd.read_csv(os.path.join(box_path, 'study_data_processed.csv'))

In [15]:
study_data.shape

(6848371, 31)

In [17]:
# Display the head of the dataset
study_data.head()

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,AGV_Yaw,AGV_Roll,AGV_spd,Timestamp,AGVname,PID,DRate,User_Relative_Speed,AGV_Relative_Speed,AGV_User_distance
0,5051.235,8610.83,-209.725,-21.416662,-92.196247,-0.159610,60.524000,-107.387000,156.022000,4990.711200,...,89.999444,0.002393,0.000238,2024-10-25 15:02:48.900,1,1,High,0.0,0.000000,6812.026513
1,5051.235,8610.83,-209.725,-21.168361,-92.384752,0.290698,60.493000,-108.133857,156.034571,4990.742143,...,89.999339,-0.000460,0.154715,2024-10-25 15:02:49.000,1,1,High,0.0,84.241150,6811.521389
2,5051.235,8610.83,-209.725,-21.240655,-92.761660,0.339485,60.271857,-108.963714,156.053429,4990.963429,...,89.999565,-0.001784,0.526054,2024-10-25 15:02:49.100,1,1,High,0.0,74.712591,6809.601293
3,5051.235,8610.83,-209.725,-21.004733,-93.120707,0.251285,59.831714,-109.665143,156.194286,4991.403286,...,90.000198,-0.001954,0.893006,2024-10-25 15:02:49.200,1,1,High,0.0,35.834089,6806.150079
4,5051.235,8610.83,-209.725,-20.717933,-91.225685,0.512156,59.295143,-110.065286,156.430571,4991.940286,...,90.001157,-0.001464,1.263035,2024-10-25 15:02:49.300,1,1,High,0.0,66.574460,6801.168018


In [19]:
print(study_data.columns)

Index(['User_X', 'User_Y', 'User_Z', 'User_Pitch', 'User_Yaw', 'User_Roll',
       'U_X', 'U_Y', 'U_Z', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z',
       'GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z', 'Confidence',
       'Gaze_on_AGV', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw',
       'AGV_Roll', 'AGV_spd', 'Timestamp', 'AGVname', 'PID', 'DRate',
       'User_Relative_Speed', 'AGV_Relative_Speed', 'AGV_User_distance'],
      dtype='object')


In [21]:
# Check for NaN values in the DataFrame
nan_counts = study_data.isna().sum()

# Display columns with NaN counts greater than 0
nan_columns = nan_counts[nan_counts > 0]
print("Columns with NaN values:")
print(nan_columns)

Columns with NaN values:
Series([], dtype: int64)


In [23]:
model_data = pd.read_csv(os.path.join(box_path, 'per_interaction_data_merged.csv'))

model_data.head()

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,AGV_User_Combination,AGV_Path_Complexity,Trust_before,Cross_First_Flag,User_Relative_Speed,AGV_Relative_Speed,Frechet_Distance_3,Frechet_Distance_5,Frechet_Distance_7,Frechet_Distance_10
0,1,High,2024-5-3,14,28,53,1,1900-01-01 15:02:48,1900-01-01 15:03:38,219,...,South - Straight,Straight,103.457506,True,74.580213,100.420508,730.13,1188.92,1532.83,1771.61
1,1,High,2024-5-3,14,28,53,2,1900-01-01 15:03:47,1900-01-01 15:04:29,0,...,North - Diagonal,Complex,211.137767,True,47.888087,232.345812,14.56,97.05,108.21,247.34
2,1,High,2024-5-3,14,28,53,3,1900-01-01 15:04:37,1900-01-01 15:05:34,268,...,South - Diagonal,Complex,211.137767,True,63.130599,120.176130,282.09,482.54,821.32,1067.08
3,1,High,2024-5-3,14,28,53,4,1900-01-01 15:05:46,1900-01-01 15:06:37,779,...,Northeast - Straight,Complex,211.137767,True,43.571998,263.284416,421.34,544.15,577.55,610.84
4,1,High,2024-5-3,14,28,53,5,1900-01-01 15:06:41,1900-01-01 15:07:44,868,...,Northwest - Straight,Complex,147.796437,True,93.720008,91.614207,486.65,852.85,1225.51,1722.14


In [25]:
print(model_data.columns)

Index(['PID', 'DRate', 'date', 'hr', 'min', 's', 'AGVname', 'StartTime',
       'EndTime', 'GazeDuration', 'mean_dist', 'min_dist', 'max_dist',
       'std_dist', 'mean_agv_spd', 'min_agv_spd', 'max_agv_spd', 'std_spd',
       'task', 'Trust', 'Safe', 'Comfort', 'Expect', 'Age', 'Gender',
       'Ethnicity', 'GamingFrequency', 'VRExperience', 'VRHeadsetExperience',
       'AGVInteraction', 'PerfectAutomation', 'TrustPropensity',
       'AutomationExperience', 'Trust1', 'Trust2', 'MWL', 'Assessment',
       'Gaze_on_AGV', 'User_Trajectory', 'AGV_Approaching',
       'AGV_User_Combination', 'AGV_Path_Complexity', 'Trust_before',
       'Cross_First_Flag', 'User_Relative_Speed', 'AGV_Relative_Speed',
       'Frechet_Distance_3', 'Frechet_Distance_5', 'Frechet_Distance_7',
       'Frechet_Distance_10'],
      dtype='object')


In [27]:
# Function to find mismatches between unique values of columns in two datasets
def check_column_mismatches(study_data, model_data, columns):
    for col in columns:
        study_unique = study_data[col].unique()
        interaction_unique = model_data[col].unique()

        # Find mismatches in both directions
        study_not_in_interaction = set(study_unique) - set(interaction_unique)
        interaction_not_in_study = set(interaction_unique) - set(study_unique)

        # Print the results for the current column
        print(f"\nColumn: {col}")
        if study_not_in_interaction:
            print(f"Values in 'study_data_processed' but not in 'per_interaction_data': {study_not_in_interaction}")
        else:
            print("No mismatches found from 'study_data_processed' to 'per_interaction_data'.")

        if interaction_not_in_study:
            print(f"Values in 'per_interaction_data' but not in 'study_data_processed': {interaction_not_in_study}")
        else:
            print("No mismatches found from 'per_interaction_data' to 'study_data_processed'.")

# List of columns to compare
columns_to_check = ['DRate', 'PID', 'AGVname']

# Check for mismatches between the two datasets
check_column_mismatches(study_data, model_data, columns_to_check)


Column: DRate
No mismatches found from 'study_data_processed' to 'per_interaction_data'.
No mismatches found from 'per_interaction_data' to 'study_data_processed'.

Column: PID
No mismatches found from 'study_data_processed' to 'per_interaction_data'.
No mismatches found from 'per_interaction_data' to 'study_data_processed'.

Column: AGVname
No mismatches found from 'study_data_processed' to 'per_interaction_data'.
No mismatches found from 'per_interaction_data' to 'study_data_processed'.


### Adding Trust, Expectancy, Safety, and Comfort as a Ground Truth Value

In [30]:
# Create a mapping dictionary from model_data
trust_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Trust'].to_dict()
expect_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Expect'].to_dict()
comfort_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Comfort'].to_dict()
safe_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Safe'].to_dict()

# Map the Trust values from model_data to study_data based on matching groups
study_data['Trust'] = study_data.set_index(['PID','AGVname', 'DRate']).index.map(trust_dict)
study_data['Expect'] = study_data.set_index(['PID','AGVname', 'DRate']).index.map(expect_dict)
study_data['Safe'] = study_data.set_index(['PID','AGVname', 'DRate']).index.map(comfort_dict)
study_data['Comfort'] = study_data.set_index(['PID','AGVname', 'DRate']).index.map(safe_dict)

# Identify the last record in each group and retain Trust value only there
last_in_group = study_data.groupby(['PID','AGVname', 'DRate']).tail(1).index

# Set Trust to NaN for all records except the last one in each group
study_data.loc[~study_data.index.isin(last_in_group), 'Trust'] = None
study_data.loc[~study_data.index.isin(last_in_group), 'Expect'] = None
study_data.loc[~study_data.index.isin(last_in_group), 'Safe'] = None
study_data.loc[~study_data.index.isin(last_in_group), 'Comfort'] = None

In [32]:
# Dictating happens in order, you can use the code below to figure out the indecies, check and see whether it has been done right
'''
# Identify and print the last record index in each group
last_in_group = study_data.groupby(['PID','AGVname', 'DRate']).tail(1).index

# Print the indices of the last elements in each group
print("Indices of the last element in each group:", last_in_group.tolist())
'''

'\n# Identify and print the last record index in each group\nlast_in_group = study_data.groupby([\'PID\',\'AGVname\', \'DRate\']).tail(1).index\n\n# Print the indices of the last elements in each group\nprint("Indices of the last element in each group:", last_in_group.tolist())\n'

### Adding AGV_Approaching Direction as a Column

In [35]:
# Create a mapping dictionary from model_data
AGV_Approaching_dict = model_data.set_index(['AGVname', 'DRate', 'PID'])['AGV_Approaching'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
study_data['AGV_Approaching'] = study_data.set_index(['AGVname', 'DRate', 'PID']).index.map(AGV_Approaching_dict)

### Adding User_Trajectory as a Column

In [37]:
# Create a mapping dictionary from model_data
User_Trajectory_dict = model_data.set_index(['AGVname', 'DRate', 'PID'])['User_Trajectory'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
study_data['User_Trajectory'] = study_data.set_index(['AGVname', 'DRate', 'PID']).index.map(User_Trajectory_dict)

In [38]:
study_data.iloc[67380:67390]

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,DRate,User_Relative_Speed,AGV_Relative_Speed,AGV_User_distance,Trust,Expect,Safe,Comfort,AGV_Approaching,User_Trajectory
67380,2073.580000,6366.038000,-209.725,-18.044760,78.766711,10.168054,49.467143,-32.217143,151.732857,2024.113143,...,High,0.000000,5.536285,9724.620341,NaN,NaN,NaN,NaN,South,Straight
67381,2073.580000,6366.038000,-209.725,-18.625317,78.488742,9.993499,49.839714,-31.996286,151.423429,2023.740571,...,High,0.000000,2.455788,9724.678076,NaN,NaN,NaN,NaN,South,Straight
67382,2073.580000,6366.038000,-209.725,-18.391520,78.446105,9.899978,50.011714,-31.924000,151.186286,2023.568429,...,High,0.000000,2.147065,9724.722139,NaN,NaN,NaN,NaN,South,Straight
67383,2073.580000,6366.038000,-209.725,-18.200067,78.242655,9.653192,49.926111,-31.871778,150.945222,2023.654111,...,High,0.000000,3.515447,9724.757589,NaN,NaN,NaN,NaN,South,Straight
67384,2073.580000,6366.038000,-209.725,-17.992152,77.972149,9.756008,49.396286,-31.797286,150.816143,2024.184000,...,High,0.000000,1.991016,9724.782600,NaN,NaN,NaN,NaN,South,Straight
67385,2073.580000,6366.038000,-209.725,-17.023103,78.176782,9.903925,48.695857,-31.692429,150.795000,2024.884286,...,High,0.000000,39128.658812,6504.786453,10.0,9.0,10.0,10.0,South,Straight
67386,2073.580000,6366.038000,-209.725,-16.250796,78.185101,10.034004,47.963429,-31.556143,150.716714,2025.616571,...,High,20112.996537,97585.241240,4501.534307,NaN,NaN,NaN,NaN,North,Diagonal
67387,2073.580000,6366.038000,-209.725,-16.896639,77.910986,10.435557,47.354714,-31.621429,150.809143,2026.225571,...,High,0.000000,93.060039,4500.205763,NaN,NaN,NaN,NaN,North,Diagonal
67388,2073.580000,6366.038000,-209.725,-17.376877,77.167135,9.856132,47.283857,-32.361000,151.384857,2026.296571,...,High,0.000000,42.997892,4497.362922,NaN,NaN,NaN,NaN,North,Diagonal
67389,2073.602714,6366.010143,-209.725,-16.489612,73.185995,9.190100,48.191000,-34.094143,152.492429,2025.392429,...,High,0.359438,54.378026,4492.978928,NaN,NaN,NaN,NaN,North,Diagonal


In [39]:
def extract_interaction_point_data(study_data, n):
    """
    Extracts a range of records around the interaction point (minimum AGV_User_distance)
    within each group defined by (PID, AGVname, DRate). Then, averages the values for numeric columns 
    and retains one entry for categorical values.

    Parameters:
    - study_data (pd.DataFrame): The input DataFrame with columns including 'PID', 'AGVname', 'DRate', 
      and 'AGV_User_distance'.
    - n (int): Hyperparameter determining the number of records above and below the interaction point 
      to include.

    Returns:
    - pd.DataFrame: A new DataFrame with the averaged values for each group.
    """
    study_data_n = pd.DataFrame()  # Empty DataFrame to hold results

    # Loop through each group
    for _, group in study_data.groupby(['PID', 'AGVname', 'DRate']):
        # Find the index of the interaction point (minimum value of 'AGV_User_distance')
        interaction_index = group['AGV_User_distance'].idxmin()

        # Define the start and end indices for slicing
        start_idx = max(interaction_index - 10 * n, group.index.min())  # Ensure we don't go out of bounds
        end_idx = min(interaction_index + 10 * n, group.index.max())    # Ensure we don't go out of bounds

        # Select the subset of the group
        subset = group.loc[start_idx:end_idx]

        # Calculate the mean of each numeric column in the subset
        numeric_means = subset.select_dtypes(include='number').mean()

        # Retain the first occurrence of each categorical value
        categorical_values = subset.select_dtypes(exclude='number').iloc[0]

        # Combine the numeric means and categorical values
        combined_row = pd.concat([numeric_means, categorical_values])

        # Append the combined row to the final DataFrame
        study_data_n = pd.concat([study_data_n, pd.DataFrame([combined_row])], ignore_index=True)

    return study_data_n

In [40]:
n_values = [3, 5, 7, 10]

# Dictionary to hold the results for each n
study_data_results = {}

# Run the function for each n and store the results
for n in n_values:
    study_data_n = extract_interaction_point_data(study_data, n)
    # Store in dictionary with key based on n
    study_data_results[f'study_data_{n}'] = study_data_n  

In [42]:
n_values = [3, 5, 7, 10]

# Create mapping dictionaries from model_data for each column
trust_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Trust'].to_dict()
expect_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Expect'].to_dict()
comfort_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Comfort'].to_dict()
safe_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Safe'].to_dict()

# Loop through each value of n and map the values, including Frechet_Distance
for n in n_values:
    study_data_n = study_data_results[f'study_data_{n}']  # Reference the current study_data_n DataFrame
    study_data_indexed = study_data_n.set_index(['PID', 'AGVname', 'DRate']).index
    
    # Map each dictionary to the corresponding column
    study_data_results[f'study_data_{n}']['Trust'] = study_data_indexed.map(trust_dict)
    study_data_results[f'study_data_{n}']['Expect'] = study_data_indexed.map(expect_dict)
    study_data_results[f'study_data_{n}']['Comfort'] = study_data_indexed.map(comfort_dict)
    study_data_results[f'study_data_{n}']['Safe'] = study_data_indexed.map(safe_dict)
    
    # Map the Frechet_Distance column for each n
    frechet_distance_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])[f'Frechet_Distance_{n}'].to_dict()
    study_data_results[f'study_data_{n}']['Frechet_Distance'] = study_data_indexed.map(frechet_distance_dict)

In [43]:
study_data_results[f'study_data_{3}'].head()

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,AGV_User_distance,Trust,Expect,Safe,Comfort,Timestamp,DRate,AGV_Approaching,User_Trajectory,Frechet_Distance
0,3237.267983,7921.798722,-209.932424,-7.012105,51.565601,-0.319141,80.245978,-52.055290,155.210223,3159.490125,...,1086.475921,10,9,10,10,2024-10-25 15:03:00.000,High,South,Straight,730.13
1,4270.979944,8160.561951,-209.725000,-15.506186,34.572302,-1.323321,90.722820,-61.930605,150.783026,4183.375256,...,744.298554,5,7,5,4,2024-10-25 14:29:30.100,Low,South,Straight,781.92
2,2042.357975,8442.231215,-209.963000,10.201758,-117.681968,-1.066347,47.060563,-81.121762,162.406487,1995.360050,...,1066.376417,10,10,10,9,2024-10-25 15:04:08.900,High,North,Diagonal,14.56
3,2029.479891,8609.023337,-209.973000,2.671027,-87.889772,1.923023,41.924204,-87.823010,161.806246,1987.554299,...,1079.245474,7,8,7,7,2024-10-25 14:32:06.000,Low,North,Diagonal,11.98
4,2513.175971,8145.066214,-209.941821,-27.887447,96.003180,5.057608,24.271096,-67.971582,153.831401,2487.223736,...,369.562853,10,10,9,10,2024-10-25 15:04:42.500,High,South,Diagonal,282.09


### Building Regression Models

In [45]:
# List of specific columns to scale
specific_columns = ['User_X', 'User_Y', 'User_Z', 'User_Pitch', 'User_Yaw', 'User_Roll',
                    'U_X', 'U_Y', 'U_Z', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z',
                    'GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z', 'Confidence',
                    'Gaze_on_AGV', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw',
                    'AGV_Roll', 'AGV_spd', 'User_Relative_Speed',
                    'AGV_Relative_Speed', 'AGV_User_distance', 'Frechet_Distance']

# List of categorical columns to apply one-hot encoding (dummies)
categorical_columns = ['AGVname', 'DRate', 'AGV_Approaching', 'User_Trajectory']

# Define list of models and their parameter grids for tuning
param_grids = {
    'LinearRegression': {},
}

# Define the models to be fine-tuned
Regressor_models = [
    ('LinearRegression', LinearRegression()),
]

# Define the header for results
header = ["Model", "MAPE", "MSE", "RMSE", "Relative RMSE", "MAE", "R-squared", "Best Params"]

# Loop through the values of n
for n in [3, 5, 7, 10]:
    # Apply get_dummies to categorical columns and include selected features, Trust, and PID
    data_with_dummies = pd.get_dummies(
        study_data_results[f'study_data_{n}'][categorical_columns + specific_columns + ['Trust', 'PID']],
        columns=categorical_columns,
        drop_first=True
    )

    # Perform the train-test split (80% train, 20% test)
    train_data, test_data = train_test_split(data_with_dummies, test_size=0.2, random_state=42)

    # Separate features (X) and target (y) for both train and test sets
    X_train = train_data.drop(columns=['Trust', 'PID'])  # Drop the target and PID for training
    y_train = train_data['Trust']

    X_test = test_data.drop(columns=['Trust', 'PID'])    # Drop the target and PID for testing
    y_test = test_data['Trust']

    # Check the shapes of the split data to verify
    print(f"\nTraining set size: {X_train.shape}, Test set size: {X_test.shape}")
    print(f"Training columns: {X_train.columns.tolist()}")

    # Initialize the scaler
    scaler = StandardScaler()

    # Scale only the specific columns in X_train
    X_train_scaled = X_train.copy()
    X_train_scaled[specific_columns] = scaler.fit_transform(X_train[specific_columns])

    # Scale only the specific columns in X_test
    X_test_scaled = X_test.copy()
    X_test_scaled[specific_columns] = scaler.transform(X_test[specific_columns])

    # Scale the target variable
    y_train_scaled = scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    y_test_scaled = scaler.transform(y_test.values.reshape(-1, 1)).flatten()

    # Define lists to store results for this iteration
    Regressor_results = []

    # Train, tune, and evaluate each regression model with GridSearchCV
    for name, model in Regressor_models:
        # If there are hyperparameters to tune, use GridSearchCV
        if name in param_grids and param_grids[name]:
            grid_search = GridSearchCV(estimator=model, param_grid=param_grids[name], cv=5, 
                                       scoring='r2', n_jobs=-1, verbose=1)
            grid_search.fit(X_train_scaled, y_train_scaled)  # Use scaled target variable
            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_
        else:
            # If no hyperparameters, just fit the model
            best_model = model
            best_model.fit(X_train_scaled, y_train_scaled)
            best_params = "N/A"

        # Make predictions
        pred = best_model.predict(X_test_scaled)
        
        # Inverse transform the predictions to get them back to the original scale
        pred_original_scale = scaler.inverse_transform(pred.reshape(-1, 1)).flatten()
        
        # Calculate evaluation metrics
        current_mape = np.mean(np.abs((y_test - pred_original_scale) / y_test)) * 100
        current_mse = mse(y_test, pred_original_scale)
        current_rmse = np.sqrt(current_mse)
        rrmse_denominator = np.mean(y_test)
        current_rrmse = current_rmse / rrmse_denominator
        current_mae = mae(y_test, pred_original_scale)
        current_r2 = r2_score(y_test, pred_original_scale)
        
        # Append results to the list
        Regressor_results.append({
            'Model': name,
            'MAPE': current_mape,
            'MSE': current_mse,
            'RMSE': current_rmse,
            'RRMSE': current_rrmse,
            'MAE': current_mae,
            'R-squared': current_r2,
            'Best Params': best_params
        })

        # Print the results
        print(tabulate([[name, f'{current_mape:.2f}', f'{current_mse:.2f}', 
                         f'{current_rmse:.2f}', f'{current_rrmse:.2f}', 
                         f'{current_mae:.2f}', f'{current_r2:.2f}', best_params]], 
                        header, tablefmt="fancy_grid"))


Training set size: (1057, 52), Test set size: (265, 52)
Training columns: ['User_X', 'User_Y', 'User_Z', 'User_Pitch', 'User_Yaw', 'User_Roll', 'U_X', 'U_Y', 'U_Z', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z', 'GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z', 'Confidence', 'Gaze_on_AGV', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd', 'User_Relative_Speed', 'AGV_Relative_Speed', 'AGV_User_distance', 'Frechet_Distance', 'AGVname_2.0', 'AGVname_3.0', 'AGVname_4.0', 'AGVname_5.0', 'AGVname_6.0', 'AGVname_7.0', 'AGVname_8.0', 'AGVname_9.0', 'AGVname_10.0', 'AGVname_11.0', 'AGVname_12.0', 'AGVname_13.0', 'AGVname_14.0', 'AGVname_15.0', 'AGVname_16.0', 'DRate_Low', 'AGV_Approaching_North', 'AGV_Approaching_Northeast', 'AGV_Approaching_Northwest', 'AGV_Approaching_South', 'AGV_Approaching_Southeast', 'AGV_Approaching_Southwest', 'AGV_Approaching_West', 'User_Trajectory_Straight']
╒══════════════════╤════════╤═══════╤════════╤═════════════════╤═══════╤═════

### We will be picking the study_data_10 as our dataset to do time series on

In [54]:
def extract_interaction_point_data(study_data, n):
    """
    Extracts a range of records around the interaction point (minimum AGV_User_distance)
    within each group defined by (PID, AGVname, DRate).

    Parameters:
    - study_data (pd.DataFrame): The input DataFrame with the columns 'PID', 'AGVname', 'DRate', and 'AGV_User_distance'.
    - n (int): Hyperparameter determining the number of records above and below the interaction point to include.

    Returns:
    - pd.DataFrame: A new DataFrame with the selected records from each group.
    """
    study_data_n = pd.DataFrame()  # Empty DataFrame to hold results

    # Loop through each group
    for _, group in study_data.groupby(['PID', 'AGVname', 'DRate']):
        # Find the index of the interaction point (minimum value of 'AGV_User_distance')
        interaction_index = group['AGV_User_distance'].idxmin()

        # Define the start and end indices for slicing
        start_idx = max(interaction_index - 10 * n, group.index.min())  # Ensure we don't go out of bounds
        end_idx = min(interaction_index + 10 * n, group.index.max())    # Ensure we don't go out of bounds

        # Select the subset of the group and append it to study_data_n
        subset = group.loc[start_idx:end_idx]
        study_data_n = pd.concat([study_data_n, subset], ignore_index=True)

    return study_data_n

study_data_10 = extract_interaction_point_data(study_data, 10)

In [56]:
# Create a mapping dictionary from model_data
trust_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Trust'].to_dict()
expect_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Expect'].to_dict()
comfort_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Comfort'].to_dict()
safe_dict = model_data.set_index(['PID', 'AGVname', 'DRate'])['Safe'].to_dict()

# Map the Trust values from model_data to study_data based on matching groups
study_data_10['Trust'] = study_data_10.set_index(['PID','AGVname', 'DRate']).index.map(trust_dict)
study_data_10['Expect'] = study_data_10.set_index(['PID','AGVname', 'DRate']).index.map(expect_dict)
study_data_10['Safe'] = study_data_10.set_index(['PID','AGVname', 'DRate']).index.map(comfort_dict)
study_data_10['Comfort'] = study_data_10.set_index(['PID','AGVname', 'DRate']).index.map(safe_dict)

# Identify the last record in each group and retain Trust value only there
last_in_group = study_data_10.groupby(['PID','AGVname', 'DRate']).tail(1).index

# Set Trust to NaN for all records except the last one in each group
study_data_10.loc[~study_data_10.index.isin(last_in_group), 'Trust'] = None
study_data_10.loc[~study_data_10.index.isin(last_in_group), 'Expect'] = None
study_data_10.loc[~study_data_10.index.isin(last_in_group), 'Safe'] = None
study_data_10.loc[~study_data_10.index.isin(last_in_group), 'Comfort'] = None

In [58]:
selected_columns = ['PID', 'AGVname', 'DRate', 'Trust', 'Expect', 'Safe', 'Comfort'] 
study_data_10.loc[200:210, selected_columns]

,PID,AGVname,DRate,Trust,Expect,Safe,Comfort
200,1,1,High,10.0,9.0,10.0,10.0
201,1,1,Low,NaN,NaN,NaN,NaN
202,1,1,Low,NaN,NaN,NaN,NaN
203,1,1,Low,NaN,NaN,NaN,NaN
204,1,1,Low,NaN,NaN,NaN,NaN
205,1,1,Low,NaN,NaN,NaN,NaN
206,1,1,Low,NaN,NaN,NaN,NaN
207,1,1,Low,NaN,NaN,NaN,NaN
208,1,1,Low,NaN,NaN,NaN,NaN
209,1,1,Low,NaN,NaN,NaN,NaN


### Adding Fretchet Distance as a column

In [ ]:
import sys
from frechetdist import frdist
from concurrent.futures import ThreadPoolExecutor
from typing import List, Tuple

In [ ]:
# Increasing the recursion limit
sys.setrecursionlimit(10000)


# Asking for the number of coordinate points in each horizon
n_points = int(input("Enter the number of coordinate points to be used in calculating the frechet distance: "))

# function to calculate Euclidean Distance
def euclidean_distance(p1, p2):
    return np.linalg.norm(p1 - p2, axis=1)

# Function to generate expected trajectory points from the current position to the end
def generate_expected_trajectory(start: Tuple[float, float], end: Tuple[float, float], num_points: int) -> np.ndarray:
    return np.column_stack((
        np.linspace(start[0], end[0], num_points),
        np.linspace(start[1], end[1], num_points)
    ))

# Function to select n points from the actual path
def select_actual_path_points(current_position: np.ndarray, path: np.ndarray, num_points: int) -> np.ndarray:
    distances = euclidean_distance(path, current_position)
    current_index = np.argmin(distances)
    end_index = min(current_index + num_points, len(path))  # Ensuring we don't go beyond the list's length
    return path[current_index:end_index]

# Fréchet Distance calculation
def frechet_distance(P: np.ndarray, Q: np.ndarray) -> float:
    ca = np.full((len(P), len(Q)), -1.0)
    return _c(ca, P, Q, len(P)-1, len(Q)-1)

def _c(ca: np.ndarray, P: np.ndarray, Q: np.ndarray, i: int, j: int) -> float:
    if ca[i, j] > -1:
        return ca[i, j]
    elif i == 0 and j == 0:
        ca[i, j] = euclidean_distance(P[0:1], Q[0:1])[0]
    elif i > 0 and j == 0:
        ca[i, j] = max(_c(ca, P, Q, i-1, 0), euclidean_distance(P[i:i+1], Q[0:1])[0])
    elif i == 0 and j > 0:
        ca[i, j] = max(_c(ca, P, Q, 0, j-1), euclidean_distance(P[0:1], Q[j:j+1])[0])
    elif i > 0 and j > 0:
        ca[i, j] = max(min(_c(ca, P, Q, i-1, j), _c(ca, P, Q, i-1, j-1), _c(ca, P, Q, i, j-1)), euclidean_distance(P[i:i+1], Q[j:j+1])[0])
    else:
        ca[i, j] = float('inf')
    return ca[i, j]

# Processing each interaction group
def process_interaction_group(group: pd.DataFrame) -> dict:
    actual_path = group[['User_X', 'User_Y']].values
    end_point = actual_path[-1]  # Endpoint to the last coordinate of the group
    frechet_distances = []

    for current_position in actual_path:
        expected_trajectory = generate_expected_trajectory(current_position, end_point, n_points)
        selected_actual_path = select_actual_path_points(current_position, actual_path, n_points)
        frechet_dist = frechet_distance(expected_trajectory, selected_actual_path)
        frechet_distances.append(frechet_dist)

    return {
        'PID': group['PID'].iloc[0],
        'AGVname': group['AGVname'].iloc[0],
        'DRate': group['DRate'].iloc[0],
        'Point_ID': group['Point_ID'].values,
        'Frechet_Distance': frechet_distances
    }

# Grouping data
grouped = study_data_10.groupby(['PID', 'AGVname', 'DRate'])

# Adding a sequential identifier within each group in the original dataset
study_data_10['Point_ID'] = study_data_10.groupby(['PID', 'AGVname', 'DRate']).cumcount()

# Creating a function to handle the parallel processing
def parallel_process_group(group_tuple):
    key, group = group_tuple
    return process_interaction_group(group)

# Using ThreadPoolExecutor for parallel processing with a limited number of threads
num_threads = 4  

with ThreadPoolExecutor(max_workers=num_threads) as executor:
    results = list(executor.map(parallel_process_group, grouped))

# Initializing an empty DataFrame for the results
frechet_df = pd.DataFrame(columns=['PID', 'AGVname', 'DRate', 'Point_ID', 'Frechet_Distance'])

# Collecting results from the parallel processing
for result in results:
    result_df = pd.DataFrame(result)
    frechet_df = pd.concat([frechet_df, result_df], ignore_index=True)

# Merging the frechet_df DataFrame with the original dataset using the keys
study_data_10 = pd.merge(study_data_10, frechet_df, on=['PID', 'AGVname', 'DRate', 'Point_ID'], how='left')

# Printing summary of the processing
print(f"Finished processing {len(grouped)} groups.")
print(f"Total points processed: {len(study_data_10)}")

# Returning the updated study_data DataFrame with Fréchet Distance
study_data_10

## Visualizing the Fretchet Distance Calculation

### The main goal of this is to confirm that all generated trajectories are directed towards 'point 3' which is the final point for each interaction

In [ ]:
# Selecting a Sample Interaction Type
sample_pid = 10  # Example PID
sample_agv_number = 5  # Example AGV Number
sample_DRate = 'High'  # Example Behavior, enter either 'High' or 'Low' 

# Filtering the dataset for the chosen interaction
sample_interaction = study_data[(study_data['PID'] == sample_pid) &
                                (study_data['AGVname'] == sample_agv_number) &
                                (study_data['DRate'] == sample_DRate)]

# Checking if the filtered interaction is empty
if sample_interaction.empty:
    print(f"No data found for PID: {sample_pid}, AGVname: {sample_agv_number}, DRate: {sample_DRate}")
else:
    # Identifying Key Points within the Interaction
    first_point = sample_interaction.iloc[0]  # First actual coordinate
    middle_point = sample_interaction.iloc[len(sample_interaction) // 3]  # Middle actual coordinate, tune as you like
    close_last_point = sample_interaction.iloc[-3]  # Modify the variable to visualize a point backwards from the last point
    final_point = sample_interaction.iloc[-1]  # Final actual coordinate, this helps visualize whether the expected trajectories are actually directed to the final point

    key_points = [first_point, middle_point, close_last_point, final_point]  # Including the final point

    # Initializing the plot
    plt.figure(figsize=(12, 8))

    # Plotting the Trajectories and Key Points
    for i, point in enumerate(key_points):
        # Plotting the actual coordinate
        plt.scatter(point['User_X'], point['User_Y'], color='blue', marker='o', zorder=5, label='Actual Coordinate' if i == 0 else "")
        plt.annotate(f'Point {i+1}', (point['User_X'], point['User_Y']), textcoords="offset points", xytext=(0,10), ha='center')

        if i < (len(key_points) - 1):  # Skip generating expected trajectory for the final point
            # Generating and plotting expected trajectory points from the current actual coordinate
            expected_trajectory = generate_expected_trajectory((point['User_X'], point['User_Y']), sample_interaction.iloc[-1][['User_X', 'User_Y']], n_points)
            plt.plot(*zip(*expected_trajectory), 'r--', label='Expected Trajectory' if i == 0 else "")

            # Plotting the actual path points used for the Fréchet Distance calculation
            actual_path_points = select_actual_path_points((point['User_X'], point['User_Y']), sample_interaction[['User_X', 'User_Y']].values, n_points)
            plt.plot(*zip(*actual_path_points), 'g-', label='Actual Path Segment' if i == 0 else "")

        # Including Fréchet Distance annotations for all points
        plt.text(point['User_X'], point['User_Y'], f'Fréchet: {point["Frechet_Distance"]:.2f}', ha='right', va='bottom')

    print(middle_point) # Allows us to confirm whether the right coordinates are being pulled, compare User_X and Y to what is displayed on the plot

    # Formatting the plot
    plt.title(f'Analysis for PID: {sample_pid}, AGV Number: {sample_agv_number}, DRate: {sample_DRate}')
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.legend(loc='best')
    plt.grid(True)
    plt.show()

In [ ]:
study_data.describe().transpose()

## Time Series Forecasting

In [64]:
study_data_10.shape

(261613, 37)

In [66]:
study_data_10.columns

Index(['User_X', 'User_Y', 'User_Z', 'User_Pitch', 'User_Yaw', 'User_Roll',
       'U_X', 'U_Y', 'U_Z', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z',
       'GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z', 'Confidence',
       'Gaze_on_AGV', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw',
       'AGV_Roll', 'AGV_spd', 'Timestamp', 'AGVname', 'PID', 'DRate',
       'User_Relative_Speed', 'AGV_Relative_Speed', 'AGV_User_distance',
       'Trust', 'Expect', 'Safe', 'Comfort', 'AGV_Approaching',
       'User_Trajectory'],
      dtype='object')

In [68]:
study_data_10.head()

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,DRate,User_Relative_Speed,AGV_Relative_Speed,AGV_User_distance,Trust,Expect,Safe,Comfort,AGV_Approaching,User_Trajectory
0,4981.397857,8586.533714,-209.725,-31.997808,47.980614,-3.696765,77.184857,-52.889143,151.637857,4906.009000,...,High,194.812458,758.046672,5393.609440,NaN,NaN,NaN,NaN,South,Straight
1,4968.832286,8573.898857,-209.725,-34.866142,47.831028,-1.422353,75.327571,-50.256429,151.159429,4895.252143,...,High,178.194613,642.388992,5316.897543,NaN,NaN,NaN,NaN,South,Straight
2,4955.818714,8561.991714,-209.725,-34.839920,47.271517,-0.272012,73.410143,-47.670714,151.813429,4884.340143,...,High,176.389652,652.921876,5239.765115,NaN,NaN,NaN,NaN,South,Straight
3,4944.972286,8552.833571,-209.725,-34.171355,46.111756,1.312510,71.386286,-45.722143,151.716571,4874.846571,...,High,141.956540,655.536138,5165.735420,NaN,NaN,NaN,NaN,South,Straight
4,4939.264000,8547.429286,-209.725,-34.814204,43.579656,3.854332,69.415286,-45.182857,151.170286,4870.281429,...,High,78.607143,661.463269,5096.457054,NaN,NaN,NaN,NaN,South,Straight


In [ ]:
study_data_10 = study_data_10.sort_values(by=['PID', 'AGVname', 'DRate']).reset_index(drop=True)

### Data Encoding

In [ ]:
# Define the columns you want to apply transformations to
plot_cols = ['User_X', 'GazeOrigin_X', 'AGV_X', 'AGV_Yaw']

# Calculate sine and cosine transformations for each column and store in new columns
for col in plot_cols:
    max_value = study_data_10[col].max()  # Get max value for each column
    # study_data_10[f'{col}_sin'] = np.sin(2 * np.pi * study_data_10[col] / max_value)
    # study_data_10[f'{col}_cos'] = np.cos(2 * np.pi * study_data_10[col] / max_value)
    plot_features[col] = study_data_10[col]/max_value
    # plot_features.index = study_data_10.index
      # Set color for 'User_X'

# Plot the sine transformations for all columns
plt.figure(figsize=(35, 20))
for i, col in enumerate(plot_cols):
    plt.subplot(len(plot_cols), 1, i + 1)
    plot_features[col].iloc[:400000].plot(subplots=True, figsize=(20, 5), color='blue')
    plt.legend(loc='upper right', fontsize=20)
    plt.ylabel(f"{col}", fontsize=20)
    plt.xlabel("Timestamp" if i == len(plot_cols) - 1 else "", fontsize=20)
    
# Define the filename with timestamp
filename = f'Cyclical_Features_Encoded.png'
full_path = os.path.join(box_path, save_dir, filename)

# Save the plots in the specified directory
plt.savefig(full_path, dpi=300)
plt.show()

In [ ]:
# Calculate sine and cosine transformations for each column and store in new columns
for col in plot_cols:
    max_value = study_data_10[col].max()  # Get max value for each column
    study_data_10[f'{col}_sin'] = np.sin(2 * np.pi * study_data_10[col] / max_value)
    study_data_10[f'{col}_cos'] = np.cos(2 * np.pi * study_data_10[col] / max_value)

# Plot the sine and cosine transformations for all columns
plt.figure(figsize=(20, len(plot_cols) * 3))  # Adjust figure height based on number of columns
for i, col in enumerate(plot_cols):
    plt.subplot(len(plot_cols), 1, i + 1)
    study_data_10[f'{col}_sin'].iloc[:20000].plot(label=f'{col}_sin', color='green')
    study_data_10[f'{col}_cos'].iloc[:20000].plot(label=f'{col}_cos', color='orange')
    plt.legend(loc='lower left', fontsize=10)
    plt.ylabel(f"{col} (sin & cos)", fontsize=10)
    plt.xlabel("Timestamp" if i == len(plot_cols) - 1 else "", fontsize=10)

# Define the filename with timestamp
filename = f'Cyclical_Features_Transformation.png'
full_path = os.path.join(box_path, save_dir, filename)

# Save the plot in the specified directory
plt.savefig(full_path, dpi=300)

# Show the plot
plt.show()

In [ ]:
study_data_10.columns

### Split the Data

In [103]:
# Define the columns you want to apply transformations to
numerical_columns = ['User_X', 'User_Y', 'User_Z', 'User_Pitch', 'User_Yaw', 'User_Roll',
                    'U_X', 'U_Y', 'U_Z', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z',
                    'GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z', 'AGV_X', 'AGV_Y', 'AGV_Z', 
                    'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd', 'User_Relative_Speed', 
                    'AGV_Relative_Speed', 'AGV_User_distance']

categorical_columns = ['AGVname', 'PID', 'DRate', 'AGV_Approaching', 'User_Trajectory', 'Timestamp']

In [107]:
# Initialize the StandardScaler and fit it on the selected columns
scaler = StandardScaler()
scaler.fit(study_data_10[selected_columns])

# Apply the scaler only to the selected columns
study_data_10_std = study_data_10.copy()
study_data_10_std[selected_columns] = scaler.transform(study_data_10[selected_columns])

In [109]:
# Convert the 'timestamp' column to datetime
study_data_10['Timestamp'] = pd.to_datetime(study_data_10['Timestamp'])

# Function to categorize time based on hour
def categorize_time(hour):
    if hour in [10, 11, 12]:
        return 'morning'
    elif hour in [13, 14, 15]:
        return 'afternoon'
    elif hour in [16, 17, 18]:
        return 'evening'
    else:
        return 'other'  # Optional for hours outside defined ranges

# Apply the function to create a new 'time_of_day' column
study_data_10_std['Timestamp'] = study_data_10['Timestamp'].dt.hour.apply(categorize_time)

In [111]:
study_data_10_std.head()

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,DRate,User_Relative_Speed,AGV_Relative_Speed,AGV_User_distance,Trust,Expect,Safe,Comfort,AGV_Approaching,User_Trajectory
0,-0.546297,1.175849,0.550577,-1.406540,0.441941,-0.820412,1.736334,-1.233855,-2.622725,-0.571054,...,High,0.547476,1.441282,3.275095,NaN,NaN,NaN,NaN,South,Straight
1,-0.549231,1.161469,0.550577,-1.593669,0.440511,-0.459130,1.706189,-1.178197,-2.643569,-0.573567,...,High,0.431039,1.096496,3.205387,NaN,NaN,NaN,NaN,South,Straight
2,-0.552270,1.147917,0.550577,-1.591958,0.435163,-0.276402,1.675069,-1.123532,-2.615076,-0.576115,...,High,0.418392,1.127896,3.135296,NaN,NaN,NaN,NaN,South,Straight
3,-0.554802,1.137494,0.550577,-1.548342,0.424078,-0.024707,1.642221,-1.082338,-2.619296,-0.578333,...,High,0.177129,1.135689,3.068025,NaN,NaN,NaN,NaN,South,Straight
4,-0.556135,1.131343,0.550577,-1.590281,0.399876,0.379053,1.610231,-1.070937,-2.643096,-0.579399,...,High,-0.266744,1.153359,3.005071,NaN,NaN,NaN,NaN,South,Straight


In [60]:
column_indices = {name: i for i, name in enumerate(study_data_10_std.columns)}

print(column_indices)

n = len(study_data_10_std)
train_data = study_data_10_std[0:int(n*0.7)]
val_data = study_data_10[int(n*0.7):int(n*0.9)]
test_data = study_data_10[int(n*0.9):]

num_features = study_data_10.shape[1]

{'User_X': 0, 'User_Y': 1, 'User_Z': 2, 'User_Pitch': 3, 'User_Yaw': 4, 'User_Roll': 5, 'U_X': 6, 'U_Y': 7, 'U_Z': 8, 'GazeOrigin_X': 9, 'GazeOrigin_Y': 10, 'GazeOrigin_Z': 11, 'GazeDirection_X': 12, 'GazeDirection_Y': 13, 'GazeDirection_Z': 14, 'Confidence': 15, 'Gaze_on_AGV': 16, 'AGV_X': 17, 'AGV_Y': 18, 'AGV_Z': 19, 'AGV_Pitch': 20, 'AGV_Yaw': 21, 'AGV_Roll': 22, 'AGV_spd': 23, 'Timestamp': 24, 'AGVname': 25, 'PID': 26, 'DRate': 27, 'User_Relative_Speed': 28, 'AGV_Relative_Speed': 29, 'AGV_User_distance': 30, 'Trust': 31, 'Expect': 32, 'Safe': 33, 'Comfort': 34, 'AGV_Approaching': 35, 'User_Trajectory': 36}


In [72]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler only on the selected columns in the training data
train_data_scaled = train_data.copy()
train_data_scaled[selected_columns] = scaler.fit_transform(train_data[selected_columns])

# Apply the same transformation to the validation and test data using the already fitted scaler
val_data_scaled = val_data.copy()
val_data_scaled[selected_columns] = scaler.transform(val_data[selected_columns])

test_data_scaled = test_data.copy()
test_data_scaled[selected_columns] = scaler.transform(test_data[selected_columns])

### Data Windowing

#### Indexes and offsets

In [115]:
class WindowGenerator():
    def __init__(self, input_width, label_width, shift,
                 train_data = train_data_scaled, val_data = val_data_scaled, test_data = test_data_scaled, label_columns=None):
        
        # Store the data.
        self.train_data = train_data
        self.val_data = val_data
        self.test_data = test_data

        # Store the label column(s).
        self.label_columns = label_columns
        if label_columns is not None:
            # Access index directly using the first item in label_columns
            self.label_column_index = train_data.columns.get_loc(label_columns[0])
            self.label_columns_indices = {name: train_data.columns.get_loc(name) for name in label_columns}
        
        # Create a column index dictionary.
        self.column_indices = {name: i for i, name in enumerate(train_data.columns)}

        # Window parameters.
        self.input_width = input_width
        self.label_width = label_width
        self.shift = shift

        self.total_window_size = input_width + shift

        self.input_slice = slice(0, input_width)
        self.input_indices = np.arange(self.total_window_size)[self.input_slice]

        self.label_start = self.total_window_size - self.label_width
        self.labels_slice = slice(self.label_start, None)
        self.label_indices = np.arange(self.total_window_size)[self.labels_slice]

    def __repr__(self):
        return '\n'.join([
            f'Total window size: {self.total_window_size}',
            f'Input indices: {self.input_indices}',
            f'Label indices: {self.label_indices}',
            f'Label column name(s): {self.label_columns}'
        ])

    def split_window(self, features):
        inputs = features[:, self.input_slice, :]
        labels = features[:, self.labels_slice, :]
        if self.label_columns is not None:
            labels = tf.stack(
                [labels[:, :, self.column_indices[name]] for name in self.label_columns],
                axis=-1)
        
        # Slicing doesn't preserve static shape information, so set the shapes
        # manually. This way the `tf.data.Datasets` are easier to inspect.
        inputs.set_shape([None, self.input_width, None])
        labels.set_shape([None, self.label_width, None])
        
        return inputs, labels
        
    WindowGenerator.split_window = split_window

w1 = WindowGenerator(input_width=200, label_width=1, shift=1,
                     label_columns=['Trust'])
w1

NameError: name 'WindowGenerator' is not defined

#### Splits

In [ ]:
# Stack three slices, the length of the total window.
example_window = tf.stack([np.array(train_data[:w1.total_window_size]),
                           np.array(train_data[100:100+w1.total_window_size]),
                           np.array(train_data[200:200+w1.total_window_size])])

example_inputs, example_labels = w1.split_window(example_window)

print('All shapes are: (batch, time, features)')
print(f'Window shape: {example_window.shape}')
print(f'Inputs shape: {example_inputs.shape}')
print(f'Labels shape: {example_labels.shape}')

## Trend of Frechet Distance and User's Relative Speed across Users

In [ ]:
# Group Data by Point_ID
grouped_for_visualization = (study_data[['Point_ID', 'Frechet_Distance', 'User_Relative_Speed']].groupby(['Point_ID']).agg(
    FD_mean = ('Frechet_Distance', 'mean'),
    FD_std = ('Frechet_Distance', 'std'),
    FD_count = ('Frechet_Distance', 'count'),
    URS_mean = ('User_Relative_Speed', 'mean'),
    URS_std = ('User_Relative_Speed', 'std'),
    URS_count = ('User_Relative_Speed', 'count'),
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for frechet distance
grouped_for_visualization['FD_ci'] = 1.96*grouped_for_visualization['FD_std'] / np.sqrt(grouped_for_visualization['FD_count'])
grouped_for_visualization['FD_ci_lower'] = grouped_for_visualization['FD_mean'] - grouped_for_visualization['FD_ci']
grouped_for_visualization['FD_ci_upper'] = grouped_for_visualization['FD_mean'] + grouped_for_visualization['FD_ci']

# Calculate the confidence intervals for user's relative speed
grouped_for_visualization['URS_ci'] = 1.96*grouped_for_visualization['URS_std'] / np.sqrt(grouped_for_visualization['URS_count'])
grouped_for_visualization['URS_ci_lower'] = grouped_for_visualization['URS_mean'] - grouped_for_visualization['FD_ci']
grouped_for_visualization['URS_ci_upper'] = grouped_for_visualization['URS_mean'] + grouped_for_visualization['FD_ci']

grouped_for_visualization.head()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 4), sharex = True)

# Plot the mean line and confidence intervals for Frechet Distance
axs[0].plot(grouped_for_visualization['Point_ID'], grouped_for_visualization['FD_mean'], label = 'Average Frechet Distance', color = 'blue')
axs[0].fill_between(grouped_for_visualization['Point_ID'], grouped_for_visualization['FD_ci_lower'], grouped_for_visualization['FD_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Frechet Distance')
axs[0].set_ylabel('Frechet Distance (Unreal Unit, 2 CM)')
axs[0].set_xlabel('Point_ID')
axs[0].set_title('Frechect Distance by Point_ID')
axs[0].legend()
axs[0].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[0].set_ylim(0, grouped_for_visualization[['FD_mean','FD_ci_upper']].max().max() + 10)
axs[0].set_xlim(0, )

# Plot the mean line and confidence intervals for User's Relative Speed
axs[1].plot(grouped_for_visualization['Point_ID'], grouped_for_visualization['URS_mean'], label = 'Average Frechet Distance', color = 'orange')
axs[1].fill_between(grouped_for_visualization['Point_ID'], grouped_for_visualization['URS_ci_lower'], grouped_for_visualization['URS_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Frechet Distance')
axs[1].set_ylabel('User Relative Speed (Unreal Unit/S, 2*(CM/S)')
axs[1].set_xlabel('Point_ID')
axs[1].set_title('User Relative Speed by Point_ID')
axs[1].legend()
axs[1].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[1].set_ylim(0, grouped_for_visualization[['FD_mean','FD_ci_upper']].max().max() + 10)
axs[1].set_xlim(0, )

plt.legend()
# Add a light gray grid
plt.grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)

# Define the filename with timestamp
filename = 'Frechet Distance and User Relative Speed Trend over time'
# plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)

plt.xticks(rotation=45)
plt.show()

## Regression for Trust 

In [ ]:
# Get dummy variables for categorical features. If 'Cross_First', and 'Frechet_Distance' were filled out, don't drop them
study_data_with_dummies = pd.get_dummies(study_data.drop(columns=['User_X_Difference','User_Y_Difference','quantile']), columns=['PID','DRate'], drop_first=True)

In [ ]:
# Splitting the dataset into training (80%) and testing (20%) sets
train_data, test_data = train_test_split(study_data_with_dummies, test_size=0.2, random_state=42)

# Checking the shape of the training and testing sets
train_data.shape, test_data.shape

In [ ]:
# Separating the independent variables (X) and the target variable (y: Trust)
X_train = train_data.drop(columns=['Trust'])
y_train = train_data['Trust']

X_test = test_data.drop(columns=['Trust'])
y_test = test_data['Trust']

print('Shape of X_train: ', X_train.shape)
print('Shape of X_test: ', X_test.shape)
print('Shape of y_train: ', y_train.shape)
print('Shape of y_test: ', y_test.shape)

In [ ]:
# Preditors are scaled but the target (y) is not touched.
# We should scale both training and test partitions.
scaler = preprocessing.MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.fit_transform(X_test), columns=X_test.columns)

In [ ]:
Regressor_models = [ 
          ('Linear Regression', LinearRegression()),
          ('KNN Regression', KNeighborsRegressor(n_neighbors=2)),
          ('RegressionTree', DecisionTreeRegressor(max_depth=4, random_state= 2024)),
          ('RandomForestRegression', RandomForestRegressor(n_estimators = 20, random_state = 2024))
        ]

# Define lists to store results
Regressor_results = []
Regressor_names = []

# Define the header separately
header = ["Model", "Mean Absolute Error", "Mean Absolute Percent Error", "Mean Squared Error", "Root Mean Squared Error", "Relative RMSE"]

# Train and evaluate each regression model
for name, model in Regressor_models:
    # Train the regression model
    fitted = model.fit(X_train_scaled, y_train)
    
    # Make predictions
    pred = fitted.predict(X_test_scaled)
    
    # Calculate Mean Absolute Percent Error (MAPE)
    current_mape = mape(y_test, pred)
    
    # Calculate Mean Squared Error (MSE)
    current_mse = mse(y_test, pred)
    
    # Calculate Root Mean Squared Error (RMSE)
    current_rmse = np.sqrt(current_mse)
    
    # Calculate Relative Root Mean Squared Error (RRMSE)
    rrmse_denominator = np.mean(y_test)
    current_rrmse = current_rmse / rrmse_denominator
    
    # Calculate Mean Absolute Error (MAE)
    current_mae = mae(y_test, pred)
    
    # Print the results
    print(tabulate([[name, f'{current_mae:.2f}', f'{current_mape:.2f}', f'{current_mse:.2f}', f'{current_rmse:.2f}', f'{current_rrmse:.2f}']], header, tablefmt="fancy_grid"))
    
    # Append results to lists
    Regressor_results.append({
        'Model': name,
        'MAE': "{:.2f}".format(current_mae),
        'MAPE': "{:.2f}".format(current_mape),
        'MSE': "{:.2f}".format(current_mse),
        'RMSE': "{:.2f}".format(current_rmse),
        'RRMSE': "{:.2f}".format(current_rrmse)
    })
    Regressor_names.append(name)

### Feature Selection
#### The results clearly indicate the models are overfitting. Thus feature selection must be done.

In [ ]:
study_data.keys()

In [ ]:
# OneHotEncoder to convert string feature to numerical
one_hot_encoder = OneHotEncoder()
encoded_feature = one_hot_encoder.fit_transform(study_data[['DRate']]).toarray()
df_encoded = pd.DataFrame(encoded_feature, columns = one_hot_encoder.get_feature_names_out(['DRate']))
study_data_encoded = study_data.join(df_encoded).drop(columns=['DRate','Trust', 'PID','quantile'], axis=1)

In [ ]:
features = study_data_encoded

scaler = preprocessing.MinMaxScaler()
scaler.fit(features)

scaled_features = scaler.transform(features)
scaled_features_df = pd.DataFrame(scaled_features, columns = features.columns)

In [ ]:
scaled_features_df.head()

In [ ]:
# Initialize PCA with the same number of components as before
pca = PCA(n_components=9)
pca_result = pca.fit_transform(scaled_features_df)
pca_df = pd.DataFrame(data = pca_result, columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9'])

# Get the explained variance ratio for each principal component
explained_variance_ratio = pca.explained_variance_ratio_

# Print the explained variance ratio for each principal component
for i, ratio in enumerate(explained_variance_ratio):
    print(f'Explained Variance Ratio for PC{i + 1}: {ratio:.2f}')

# Calculate the cumulative explained variance ratio
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

# Print the cumulative explained variance ratio
print('\nCumulative Explained Variance Ratio:')
for i, ratio in enumerate(cumulative_variance_ratio):
    print(f'PC{i + 1}: {ratio:.2f}')

# Determine which features to keep based on the cumulative explained variance ratio
num_components_to_keep = np.argmax(cumulative_variance_ratio >= 0.95) + 1
print(f'\nNumber of components to keep for 95% variance: {num_components_to_keep}')

# Determine which features contribute the most to each principal component
top_features_per_component = {}
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

for pc, loading in enumerate(loadings.T):
    top_features = scaled_features_df.columns[np.argsort(np.abs(loading))[::-1]][:3]  # Top 3 features per component
    top_features_per_component[f'PC{pc + 1}'] = top_features.tolist()

# Plot the explained variance and cumulative explained variance
plt.figure(figsize=(8, 6))
plt.plot(np.arange(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', label='Explained Variance Ratio')
plt.plot(np.arange(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, marker='s', label='Cumulative Explained Variance Ratio')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance Threshold')

# Annotate the plot with the top contributing features for each principal component
for pc, top_features in top_features_per_component.items():
    annotation_text = f'{pc}: {", ".join(top_features)}'
    plt.annotate(annotation_text, xy=(int(pc[2:]), 0.95), xytext=(int(pc[2:]), 0.7 - int(pc[2:]) * 0.06)
                 #,arrowprops=dict(facecolor='black', arrowstyle='<-')
                 ,fontsize=10, ha='center')

plt.xlabel('Number of Components')
plt.ylabel('Variance Ratio')
plt.title('Explained Variance and Cumulative Explained Variance')
plt.legend(loc='upper left', bbox_to_anchor=(1, 0.5))
plt.grid(True)
plt.tight_layout()

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename with timestamp
filename = f"Explained Variance and Cumulative Explained Variance_{timestamp}.png"

plt.savefig(save_dir + filename, dpi = 600)

plt.show()

### Running Regression Models with New PCs

In [ ]:
X_of_pca = pca_df[['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9']]
y_of_pca = study_data['Trust']

X_of_pca_train, X_of_pca_test, y_of_pca_train, y_of_pca_test = train_test_split(X_of_pca, y_of_pca, test_size=0.2, random_state=2024)

In [ ]:
Regressor_models = [ 
          ('Linear Regression', LinearRegression()),
          ('KNN Regression', KNeighborsRegressor(n_neighbors=2)),
          ('RegressionTree', DecisionTreeRegressor(max_depth=4, random_state= 2024)),
          ('RandomForestRegression', RandomForestRegressor(n_estimators = 20, random_state = 2024))
        ]

# Define lists to store results
Regressor_results = []
Regressor_names = []

# Define the header separately
header = ["Model", "Mean Absolute Error", "Mean Absolute Percent Error", "Mean Squared Error", "Root Mean Squared Error", "Relative RMSE"]

# Train and evaluate each regression model
for name, model in Regressor_models:
    # Train the regression model
    fitted = model.fit(X_of_pca_train, y_of_pca_train)
    
    # Make predictions
    pred = fitted.predict(X_of_pca_test)

    # Calculate Mean Absolute Error (MAE)
    current_mae = mae(y_of_pca_test, pred)
    
    # Calculate Mean Absolute Percent Error (MAPE)
    current_mape = mape(y_of_pca_test, pred)
    
    # Calculate Mean Squared Error (MSE)
    current_mse = mse(y_of_pca_test, pred)
    
    # Calculate Root Mean Squared Error (RMSE)
    current_rmse = np.sqrt(current_mse)
    
    # Calculate Relative Root Mean Squared Error (RRMSE)
    rrmse_denominator = np.mean(y_of_pca_test)
    current_rrmse = current_rmse / rrmse_denominator
    
    # Print the results
    print(tabulate([[name, f'{current_mae:.2f}', f'{current_mape:.2f}', f'{current_mse:.2f}', f'{current_rmse:.2f}', f'{current_rrmse:.2f}']], header, tablefmt="fancy_grid"))
    
    # Append results to lists
    Regressor_results.append({
        'Model': name,
        'MAE': "{:.2f}".format(current_mae),
        'MAPE': "{:.2f}".format(current_mape),
        'MSE': "{:.2f}".format(current_mse),
        'RMSE': "{:.2f}".format(current_rmse),
        'RRMSE': "{:.2f}".format(current_rrmse)
    })
    Regressor_names.append(name)

## Interpolating the Trust Value based on Features

In [ ]:
# Sort the dataset
study_data_sorted = study_data.sort_values(by=['PID', 'AGVname', 'DRate', 'Point_ID'], ignore_index=True)
study_data_sorted.loc[:,['PID', 'AGVname', 'DRate', 'Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head()
# study_data_copied = study_data_sorted.copy()
# study_data_copied['Trust'] = np.nan

In [ ]:
# Group the data
grouped_study_data_sorted = study_data_sorted.groupby(['PID', 'AGVname', 'DRate'])
study_data_sorted.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head()

In [ ]:
# calculating the length of the series 
#len = grouped_study_data_sorted['Trust'].size 
# Loop over the groups
for (pid, agvname, drate), group in grouped_study_data_sorted:
    indices = group.index.tolist()
    #trust_values = group['Trust'].values
    for idx in indices[1:-1]:
         study_data_sorted.at[idx, 'Trust'] = np.nan
        
    #first_index = study_data_sorted.at[indices[0], 'Trust']
    #print(first_index)
    #last_index = study_data_sorted.at[indices[-1], 'Trust']
    #print(last_index)

    #study_data_sorted['Trust'] = np.nan
    
    #study_data_sorted.at[indices[0], 'Trust']  = first_index
    #study_data_sorted.at[indices[-1], 'Trust'] = last_index
study_data_sorted.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Linear Interpolation

In [ ]:
study_data_linearly_interpolated = study_data_sorted.interpolate(method='linear', limit_direction='forward', axis=0)
study_data_linearly_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### SLinear Interpolation

In [ ]:
study_data_slinearly_interpolated = study_data_sorted.interpolate(method='slinear', limit_direction='forward', axis=0)
study_data_slinearly_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Quadratic Interpolation

In [ ]:
study_data_quadratically_interpolated = study_data_sorted.interpolate(method='quadratic', order=2)
study_data_quadratically_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Cubic Interpolation

In [ ]:
study_data_cubicly_interpolated = study_data_sorted.interpolate(method='cubic', order=3)
study_data_cubicly_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Polynomial Interpolation

In [ ]:
study_data_polynomial_interpolated = study_data_sorted.interpolate(method='polynomial', order=5)
study_data_polynomial_interpolated.loc[:,['Point_ID','Trust', 'Safe', 'Comfort', 'Expect']].head(180)

### Visualizing the Interpolation

In [ ]:
# Group Data by Point_ID
grouped_for_study_data_linearly_interpolated = (study_data_linearly_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_linearly_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_linearly_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_linearly_interpolated['Trust_count'])
grouped_for_study_data_linearly_interpolated['Trust_ci_lower'] = grouped_for_study_data_linearly_interpolated['Trust_mean'] - grouped_for_study_data_linearly_interpolated['Trust_ci']
grouped_for_study_data_linearly_interpolated['Trust_ci_upper'] = grouped_for_study_data_linearly_interpolated['Trust_mean'] + grouped_for_study_data_linearly_interpolated['Trust_ci']

# Group Data by Point_ID
grouped_for_study_data_slinearly_interpolated = (study_data_slinearly_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_slinearly_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_slinearly_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_slinearly_interpolated['Trust_count'])
grouped_for_study_data_slinearly_interpolated['Trust_ci_lower'] = grouped_for_study_data_slinearly_interpolated['Trust_mean'] - grouped_for_study_data_slinearly_interpolated['Trust_ci']
grouped_for_study_data_slinearly_interpolated['Trust_ci_upper'] = grouped_for_study_data_slinearly_interpolated['Trust_mean'] + grouped_for_study_data_slinearly_interpolated['Trust_ci']

# Group Data by Point_ID
grouped_for_study_data_quadratically_interpolated = (study_data_quadratically_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_quadratically_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_quadratically_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_quadratically_interpolated['Trust_count'])
grouped_for_study_data_quadratically_interpolated['Trust_ci_lower'] = grouped_for_study_data_quadratically_interpolated['Trust_mean'] - grouped_for_study_data_quadratically_interpolated['Trust_ci']
grouped_for_study_data_quadratically_interpolated['Trust_ci_upper'] = grouped_for_study_data_quadratically_interpolated['Trust_mean'] + grouped_for_study_data_quadratically_interpolated['Trust_ci']

# Group Data by Point_ID
grouped_for_study_data_cubicly_interpolated = (study_data_cubicly_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_cubicly_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_cubicly_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_cubicly_interpolated['Trust_count'])
grouped_for_study_data_cubicly_interpolated['Trust_ci_lower'] = grouped_for_study_data_cubicly_interpolated['Trust_mean'] - grouped_for_study_data_cubicly_interpolated['Trust_ci']
grouped_for_study_data_cubicly_interpolated['Trust_ci_upper'] = grouped_for_study_data_cubicly_interpolated['Trust_mean'] + grouped_for_study_data_cubicly_interpolated['Trust_ci']

# Group Data by Point_ID
grouped_for_study_data_polynomial_interpolated = (study_data_polynomial_interpolated[['Point_ID', 'Trust']].groupby(['Point_ID']).agg(
    Trust_mean = ('Trust', 'mean'),
    Trust_std = ('Trust', 'std'),
    Trust_count = ('Trust', 'count')
    )).reset_index()

# grouped_for_visualization = grouped_for_visualization.droplevel(axis = 1, level =0).reset_index()

# Calculate the confidence intervals for trust
grouped_for_study_data_polynomial_interpolated['Trust_ci'] = 1.96*grouped_for_study_data_polynomial_interpolated['Trust_std'] / np.sqrt(grouped_for_study_data_polynomial_interpolated['Trust_count'])
grouped_for_study_data_polynomial_interpolated['Trust_ci_lower'] = grouped_for_study_data_polynomial_interpolated['Trust_mean'] - grouped_for_study_data_polynomial_interpolated['Trust_ci']
grouped_for_study_data_polynomial_interpolated['Trust_ci_upper'] = grouped_for_study_data_polynomial_interpolated['Trust_mean'] + grouped_for_study_data_polynomial_interpolated['Trust_ci']

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(32, 4), sharex = True)

# Plot the mean line and confidence intervals for linearly interpolated trust
axs[0].plot(grouped_for_study_data_linearly_interpolated['Point_ID'], grouped_for_study_data_linearly_interpolated['Trust_mean'], label = 'Average Trust', color = 'orange')
axs[0].fill_between(grouped_for_study_data_linearly_interpolated['Point_ID'], grouped_for_study_data_linearly_interpolated['Trust_ci_lower'], grouped_for_study_data_linearly_interpolated['Trust_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Trust')
axs[0].set_ylabel('Trust')
axs[0].set_xlabel('Point_ID')
axs[0].set_title('Linearly Interpolated Trust by Point_ID')
axs[0].legend()
axs[0].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[0].set_ylim(grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_linearly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[0].set_xlim(0, )

# Plot the mean line and confidence intervals for slinearly interpolated trust
axs[1].plot(grouped_for_study_data_slinearly_interpolated['Point_ID'], grouped_for_study_data_slinearly_interpolated['Trust_mean'], label = 'Average Trust', color = 'orange')
axs[1].fill_between(grouped_for_study_data_slinearly_interpolated['Point_ID'], grouped_for_study_data_slinearly_interpolated['Trust_ci_lower'], grouped_for_study_data_slinearly_interpolated['Trust_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Trust')
axs[1].set_ylabel('Trust')
axs[1].set_xlabel('Point_ID')
axs[1].set_title('SLinearly Interpolated Trust by Point_ID')
axs[1].legend()
axs[1].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[1].set_ylim(grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_slinearly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[1].set_xlim(0, )

# Plot the mean line and confidence intervals for quadratically interpolated trust
axs[2].plot(grouped_for_study_data_quadratically_interpolated['Point_ID'], grouped_for_study_data_quadratically_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[2].fill_between(grouped_for_study_data_quadratically_interpolated['Point_ID'], grouped_for_study_data_quadratically_interpolated['Trust_ci_lower'], grouped_for_study_data_quadratically_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[2].set_ylabel('Trust')
axs[2].set_xlabel('Point_ID')
axs[2].set_title('Quadratically (O=2) Interpolated Trust by Point_ID')
axs[2].legend()
axs[2].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[2].set_ylim(grouped_for_study_data_quadratically_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_quadratically_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[2].set_xlim(0, )

# Plot the mean line and confidence intervals for cubicly interpolated trust
axs[3].plot(grouped_for_study_data_cubicly_interpolated['Point_ID'], grouped_for_study_data_cubicly_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[3].fill_between(grouped_for_study_data_cubicly_interpolated['Point_ID'], grouped_for_study_data_cubicly_interpolated['Trust_ci_lower'], grouped_for_study_data_cubicly_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[3].set_ylabel('Trust')
axs[3].set_xlabel('Point_ID')
axs[3].set_title('Cubicly (O=3) Interpolated Trust by Point_ID')
axs[3].legend()
axs[3].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[3].set_ylim(grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[3].set_xlim(0, )

# Plot the mean line and confidence intervals for polynomial interpolated trust
axs[4].plot(grouped_for_study_data_polynomial_interpolated['Point_ID'], grouped_for_study_data_polynomial_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[4].fill_between(grouped_for_study_data_polynomial_interpolated['Point_ID'], grouped_for_study_data_polynomial_interpolated['Trust_ci_lower'], grouped_for_study_data_polynomial_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[4].set_ylabel('Trust')
axs[4].set_xlabel('Point_ID')
axs[4].set_title('Polynomial (O=5) Interpolated Trust by Point_ID')
axs[4].legend()
axs[4].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[4].set_ylim(grouped_for_study_data_polynomial_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_polynomial_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[4].set_xlim(0, )

plt.legend()
# Add a light gray grid
plt.grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)

# Define the filename with timestamp
filename = 'Interpolated Trust over Time with the same ylim'
plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)

plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(32, 4), sharex = True)

# Plot the mean line and confidence intervals for linearly interpolated trust
axs[0].plot(grouped_for_study_data_linearly_interpolated['Point_ID'], grouped_for_study_data_linearly_interpolated['Trust_mean'], label = 'Average Trust', color = 'orange')
axs[0].fill_between(grouped_for_study_data_linearly_interpolated['Point_ID'], grouped_for_study_data_linearly_interpolated['Trust_ci_lower'], grouped_for_study_data_linearly_interpolated['Trust_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Trust')
axs[0].set_ylabel('Trust')
axs[0].set_xlabel('Point_ID')
axs[0].set_title('Linearly Interpolated Trust by Point_ID')
axs[0].legend()
axs[0].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[0].set_ylim(grouped_for_study_data_linearly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_linearly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[0].set_xlim(0, )

# Plot the mean line and confidence intervals for slinearly interpolated trust
axs[1].plot(grouped_for_study_data_slinearly_interpolated['Point_ID'], grouped_for_study_data_slinearly_interpolated['Trust_mean'], label = 'Average Trust', color = 'orange')
axs[1].fill_between(grouped_for_study_data_slinearly_interpolated['Point_ID'], grouped_for_study_data_slinearly_interpolated['Trust_ci_lower'], grouped_for_study_data_slinearly_interpolated['Trust_ci_upper'], color='orange', alpha = 0.15, label = '95% CI for Trust')
axs[1].set_ylabel('Trust')
axs[1].set_xlabel('Point_ID')
axs[1].set_title('SLinearly Interpolated Trust by Point_ID')
axs[1].legend()
axs[1].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[1].set_ylim(grouped_for_study_data_slinearly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_slinearly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[1].set_xlim(0, )

# Plot the mean line and confidence intervals for quadratically interpolated trust
axs[2].plot(grouped_for_study_data_quadratically_interpolated['Point_ID'], grouped_for_study_data_quadratically_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[2].fill_between(grouped_for_study_data_quadratically_interpolated['Point_ID'], grouped_for_study_data_quadratically_interpolated['Trust_ci_lower'], grouped_for_study_data_quadratically_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[2].set_ylabel('Trust')
axs[2].set_xlabel('Point_ID')
axs[2].set_title('Quadratically (O=2) Interpolated Trust by Point_ID')
axs[2].legend()
axs[2].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[2].set_ylim(grouped_for_study_data_quadratically_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_quadratically_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[2].set_xlim(0, )

# Plot the mean line and confidence intervals for cubicly interpolated trust
axs[3].plot(grouped_for_study_data_cubicly_interpolated['Point_ID'], grouped_for_study_data_cubicly_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[3].fill_between(grouped_for_study_data_cubicly_interpolated['Point_ID'], grouped_for_study_data_cubicly_interpolated['Trust_ci_lower'], grouped_for_study_data_cubicly_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[3].set_ylabel('Trust')
axs[3].set_xlabel('Point_ID')
axs[3].set_title('Cubicly (O=3) Interpolated Trust by Point_ID')
axs[3].legend()
axs[3].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[3].set_ylim(grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_cubicly_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[3].set_xlim(0, )

# Plot the mean line and confidence intervals for polynomial interpolated trust
axs[4].plot(grouped_for_study_data_polynomial_interpolated['Point_ID'], grouped_for_study_data_polynomial_interpolated['Trust_mean'], label = 'Average Trust', color = 'blue')
axs[4].fill_between(grouped_for_study_data_polynomial_interpolated['Point_ID'], grouped_for_study_data_polynomial_interpolated['Trust_ci_lower'], grouped_for_study_data_polynomial_interpolated['Trust_ci_upper'], color='blue', alpha = 0.15, label = '95% CI for Trust')
axs[4].set_ylabel('Trust')
axs[4].set_xlabel('Point_ID')
axs[4].set_title('Polynomial (O=5) Interpolated Trust by Point_ID')
axs[4].legend()
axs[4].grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)
axs[4].set_ylim(grouped_for_study_data_polynomial_interpolated[['Trust_mean','Trust_ci_lower']].min().min(), grouped_for_study_data_polynomial_interpolated[['Trust_mean','Trust_ci_upper']].max().max() + 10)
axs[4].set_xlim(0, )

plt.legend()
# Add a light gray grid
plt.grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.5)

# Define the filename with timestamp
filename = 'Interpolated Trust over Time with the different ylim'
plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)

plt.xticks(rotation=45)
plt.show()